# [**Brazil Air Traffic Data 2000–2025 (ANAC)**](https://www.kaggle.com/datasets/sturarods/anac-national-civil-aviation-agency-2000-2025)
![image.png](https://i0.wp.com/passageirodeprimeira.com/wp-content/uploads/2024/06/design-sem-nome-17.png?resize=1200%2C627&ssl=1)

## Imports

In [125]:
# Importtando as bibliotecas necessárias pára o projeto (Regressão)
from pyspark.sql import SparkSession
from pyspark.sql.functions import col
# MLlib (Biblioteca escalável de Machine Learning)
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, DecisionTreeRegressor
from pyspark.ml import Pipeline
from pyspark.ml.evaluation import RegressionEvaluator

## Spark

In [5]:
# Incialização do Spark
spark = SparkSession.builder \
    .appName("ANAC Regressao") \
    .master("spark://spark-master:7077") \
    .getOrCreate()

spark

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/05/03 16:30:06 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Lendo o arquivo CSV

In [6]:
# Lendo o dataset
df = spark.read.csv("/opt/spark-data/raw/anac_brazil.csv", header=True, inferSchema=True)

## Análises Iniciais

In [7]:
# Vamos confirmar se o Dataframe é do Spark mesmo
print(type(df))

<class 'pyspark.sql.dataframe.DataFrame'>


In [8]:
# Vamos exibir as colunas existente no dataset com seus tipos
df.printSchema()

root
 |-- id_basica: double (nullable = true)
 |-- id_empresa: integer (nullable = true)
 |-- sg_empresa_icao: string (nullable = true)
 |-- sg_empresa_iata: string (nullable = true)
 |-- nm_empresa: string (nullable = true)
 |-- nm_pais: string (nullable = true)
 |-- ds_tipo_empresa: string (nullable = true)
 |-- nr_voo: integer (nullable = true)
 |-- nr_singular: string (nullable = true)
 |-- id_di: integer (nullable = true)
 |-- cd_di: string (nullable = true)
 |-- ds_di: string (nullable = true)
 |-- ds_grupo_di: string (nullable = true)
 |-- dt_referencia: date (nullable = true)
 |-- nr_ano_referencia: integer (nullable = true)
 |-- nr_semestre_referencia: integer (nullable = true)
 |-- nm_semestre_referencia: string (nullable = true)
 |-- nr_trimestre_referencia: integer (nullable = true)
 |-- nm_trimestre_referencia: string (nullable = true)
 |-- nr_mes_referencia: integer (nullable = true)
 |-- nm_mes_referencia: string (nullable = true)
 |-- nr_semana_referencia: integer (null

In [9]:
# Convertendo para parquet
df.write.mode("overwrite").parquet("/opt/spark-data/processed/anac_parquet")

26/05/03 16:33:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
                                                                                

In [10]:
# Lendo o parquet
df_parquet = spark.read.parquet("/opt/spark-data/processed/anac_parquet")
# Vamos ver os primeiros 5 linhas do dataset
df_parquet.show(5)

+-----------+----------+---------------+---------------+--------------------+-------+--------------------+------+-----------+-----+-----+--------------------+-----------+-------------+-----------------+----------------------+----------------------+-----------------------+-----------------------+-----------------+-----------------+--------------------+------------------------+-----------------+---------------------+-------------+-------------+---------------+----------------------+---------------------+-----------------+-------------------+---------------+-------------------+------------------------+------------------------+-------------------------+-------------------------+-------------------+-------------------+----------------------+--------------------------+-------------------+-----------------------+-------------------+--------------+--------------+--------------------+-------------------+------------+----------------+--------------+--------------------+--------+---------------+-

In [11]:
num_linhas = df_parquet.count()
print(f"Quantidade de linhas no dataset = {num_linhas}")

Quantidade de linhas no dataset = 22120241


**Obs.: Então temos um dataset grande com 111 colunas e mais de 22 milhões de linhas.**

In [12]:
df_parquet.describe().show()

26/05/03 16:59:35 WARN DAGScheduler: Broadcasting large task binary with size 1000.9 KiB
[Stage 10:>                                                         (0 + 1) / 1]

+-------+--------------------+------------------+---------------+-------------------+-----------------+-------------+--------------------+------------------+-----------------+------------------+------------------+--------------------+-----------+------------------+----------------------+----------------------+-----------------------+-----------------------+------------------+-----------------+--------------------+------------------------+------------------+---------------------+------------------+-------------+-------------------+----------------------+---------------------+-----------------+-------------------+------------------------+------------------------+-------------------------+-------------------------+-------------------+-------------------+----------------------+--------------------------+-------------------+-----------------------+-------------------+--------------+--------------+--------------------+--------------------+------------+----------------+--------------+--------

### Verficando se tem colunas com valores nulos

In [15]:
# Criando uma tabela temporária
df_parquet.createOrReplaceTempView("anac_raw")

In [16]:
cols = df_parquet.columns

query = "SELECT\n  " + ",\n  ".join([
    f"SUM(CASE WHEN {c} IS NULL THEN 1 ELSE 0 END) AS {c}_nulls"
    for c in cols
]) + "\nFROM anac_raw"

spark.sql(query).show(truncate=False, vertical=True)

[Stage 11:====================================================>   (16 + 1) / 17]

-RECORD 0------------------------------------
 id_basica_nulls                  | 0        
 id_empresa_nulls                 | 0        
 sg_empresa_icao_nulls            | 0        
 sg_empresa_iata_nulls            | 559947   
 nm_empresa_nulls                 | 0        
 nm_pais_nulls                    | 184      
 ds_tipo_empresa_nulls            | 0        
 nr_voo_nulls                     | 0        
 nr_singular_nulls                | 224985   
 id_di_nulls                      | 0        
 cd_di_nulls                      | 2        
 ds_di_nulls                      | 2        
 ds_grupo_di_nulls                | 2        
 dt_referencia_nulls              | 0        
 nr_ano_referencia_nulls          | 0        
 nr_semestre_referencia_nulls     | 0        
 nm_semestre_referencia_nulls     | 0        
 nr_trimestre_referencia_nulls    | 0        
 nm_trimestre_referencia_nulls    | 0        
 nr_mes_referencia_nulls          | 0        
 nm_mes_referencia_nulls          

### Vericando se tem dados duplicados

In [25]:
spark.sql("""
SELECT id_basica, COUNT(*) as qtd_duplicdos
FROM anac_raw
GROUP BY id_basica
HAVING COUNT(*) > 1
""").show()

[Stage 37:==============>                                          (3 + 9) / 12]

+---------+-------------+
|id_basica|qtd_duplicdos|
+---------+-------------+
+---------+-------------+



**Obs.: Como a coluna id_basica é de chave única, é mais fácil verificar só essa coluna para ver se tem linhas duplicados, ao invés de verificar todas as colunas.**<br>
**Nesse dataset não tem dados duplicados.**

# **Análise Exploratória de Dados**

**A variável alvo será a feature ""nr_passag_pagos""(Double): Número de passageiros que ocupam assentos comercializados ao público e que geram receita, com a compra de assentos, para a empresa de transporte aéreo.**

In [21]:
# Vamos ver a distribuição de valores da variavel alvo
df_parquet.select("nr_passag_pagos").describe().show()

[Stage 21:=======================================>                (12 + 5) / 17]

+-------+------------------+
|summary|   nr_passag_pagos|
+-------+------------------+
|  count|          22120214|
|   mean|101.52640417493248|
| stddev| 60.44896348420348|
|    min|               0.0|
|    max|            1392.0|
+-------+------------------+



In [28]:
spark.sql("""
SELECT 
    FLOOR(nr_passag_pagos / 100) * 100 as faixa,
    COUNT(*) as qtd
FROM anac_raw
GROUP BY faixa
ORDER BY faixa
""").show()

[Stage 41:====================================>                   (11 + 6) / 17]

+-----+--------+
|faixa|     qtd|
+-----+--------+
| NULL|      27|
|    0|10613247|
|  100|10512309|
|  200|  856802|
|  300|  124755|
|  400|   12419|
|  500|     542|
|  600|      87|
|  700|      25|
|  800|      12|
|  900|       5|
| 1000|       8|
| 1300|       3|
+-----+--------+



**Obs.: Aqui percebe-se que temos poucos valores que são acima de 600, esses então podem ser considerados outliers dentre as quantidades de valores menores na coluna nr_passag_pagos.**

In [36]:
# Faixa de valores por cada ano (2000-2025)
spark.sql("""
SELECT 
    nr_ano_referencia,
    FLOOR(nr_passag_pagos / 100) * 100 as faixa,
    COUNT(*) as qtd
FROM anac_raw
WHERE nr_ano_referencia BETWEEN 2000 AND 2025
GROUP BY nr_ano_referencia, faixa
ORDER BY nr_ano_referencia, faixa
""").show(200)

+-----------------+-----+------+
|nr_ano_referencia|faixa|   qtd|
+-----------------+-----+------+
|             2000|    0|654590|
|             2000|  100|116643|
|             2000|  200| 13244|
|             2000|  300|  1150|
|             2000|  400|    81|
|             2000|  500|     6|
|             2000|  600|     2|
|             2001|    0|692815|
|             2001|  100|117370|
|             2001|  200| 10334|
|             2001|  300|   394|
|             2001|  400|    27|
|             2001|  500|     4|
|             2001|  600|     2|
|             2001|  700|     2|
|             2001|  900|     1|
|             2002|    0|628967|
|             2002|  100|118153|
|             2002|  200| 12525|
|             2002|  300|   492|
|             2002|  400|   184|
|             2002|  500|    45|
|             2002|  600|    19|
|             2002|  700|     9|
|             2002|  800|     1|
|             2003|    0|466232|
|             2003|  100|139691|
|         

In [40]:
# Média de passageiros pagos por ano (2000-2025)
spark.sql("""
SELECT 
    nr_ano_referencia,
    AVG(nr_passag_pagos) as media_passageiros
FROM anac_raw
GROUP BY nr_ano_referencia
ORDER BY nr_ano_referencia
""").show(26)

+-----------------+------------------+
|nr_ano_referencia| media_passageiros|
+-----------------+------------------+
|             2000| 56.81763385243523|
|             2001| 56.82681506402955|
|             2002| 60.45451114223528|
|             2003| 69.23655882788982|
|             2004|  76.1667893198703|
|             2005| 85.13379517850797|
|             2006| 88.93861455304872|
|             2007| 88.66787287454498|
|             2008| 90.74354123666954|
|             2009| 91.20481242488411|
|             2010| 94.79217089340209|
|             2011| 97.44918227643119|
|             2012|100.02100475504966|
|             2013|104.72000152781996|
|             2014|111.46910315570314|
|             2015|112.20833900462777|
|             2016|115.51374516731927|
|             2017|120.66119062921138|
|             2018|121.97702925700992|
|             2019|126.19554000033452|
|             2020| 110.6200781854534|
|             2021|111.89166670792957|
|             2022|117.23

<u><strong>Análises sobre os valores da coluna nr_passag_pagos</strong></u>

- **Observa-se um crescimento consistente no número de passageiros pagos ao longo dos anos, evidenciado pelo aumento da frequência nas faixas superiores (especialmente faixa 100).**

- **O comportamento pode estar alinhado com o crescimento do setor aéreo brasileiro, impulsionado por fatores econômicos e aumento do acesso ao transporte aéreo.**

- **Em 2020, há uma queda significativa em todas as faixas, refletindo diretamente o impacto da pandemia de COVID-19, que reduziu drasticamente a demanda por voos.**

- **A partir de 2022, observa-se uma recuperação progressiva, com retomada do crescimento até 2025.**

In [43]:
# Média vs Mediana, para ver se tem outliers (2000-2025)
spark.sql("""
SELECT 
    nr_ano_referencia,
    AVG(nr_passag_pagos) as media,
    percentile(nr_passag_pagos, 0.5) as mediana
FROM anac_raw
GROUP BY nr_ano_referencia
ORDER BY nr_ano_referencia
""").show(26)

[Stage 86:====================================>                   (11 + 6) / 17]

+-----------------+------------------+-------+
|nr_ano_referencia|             media|mediana|
+-----------------+------------------+-------+
|             2000| 56.81763385243523|   46.0|
|             2001| 56.82681506402955|   48.0|
|             2002| 60.45451114223528|   52.0|
|             2003| 69.23655882788982|   62.0|
|             2004|  76.1667893198703|   72.0|
|             2005| 85.13379517850797|   86.0|
|             2006| 88.93861455304872|   91.0|
|             2007| 88.66787287454498|   90.0|
|             2008| 90.74354123666954|   90.0|
|             2009| 91.20481242488411|   89.0|
|             2010| 94.79217089340209|   95.0|
|             2011| 97.44918227643119|   98.0|
|             2012|100.02100475504966|  101.0|
|             2013|104.72000152781996|  104.0|
|             2014|111.46910315570314|  112.0|
|             2015|112.20833900462777|  111.0|
|             2016|115.51374516731927|  114.0|
|             2017|120.66119062921138|  120.0|
|            

**Obs.: Aqui pode-se ver que a média está bem próximo da mediana, isso se deve ao fato de que os valores grandes são poucos comparados a quantidade de valores menores. Mas em alguns anos a média ultrapassa a mediana mesmo que pouco, podendo evidenciar a presença mais forte de outliers (2000 - 2004).**

In [45]:
# Taxa de ocupação dos voos (2000-2025)
spark.sql("""
SELECT 
    nr_ano_referencia,
    AVG(nr_passag_pagos / nr_assentos_ofertados) as taxa_ocupacao
FROM anac_raw
GROUP BY nr_ano_referencia
ORDER BY nr_ano_referencia
""").show(26)

+-----------------+------------------+
|nr_ano_referencia|     taxa_ocupacao|
+-----------------+------------------+
|             2000|0.5490681926781859|
|             2001|0.5425218379216729|
|             2002|0.5375020795831918|
|             2003|0.5750967213326418|
|             2004|0.6070987664048755|
|             2005| 0.648344078947748|
|             2006|0.6611376967177991|
|             2007|0.6337574338393833|
|             2008|0.6280575083820142|
|             2009|0.6347976634067656|
|             2010|0.6575845395176508|
|             2011|0.6843057898804507|
|             2012|0.7031744327980083|
|             2013|0.7265857788950906|
|             2014|0.7642059427511854|
|             2015|0.7615520369944752|
|             2016|0.7694952056249953|
|             2017|0.7895316495620626|
|             2018|0.7853305799501306|
|             2019|0.8025087651300616|
|             2020|0.7533561857054184|
|             2021|0.7747577487700494|
|             2022|0.7631

<u><strong>Análises sobre as taxas de ocupação ao longo dos anos (2000 - 2025)</strong></u>

- **A análise da taxa de ocupação evidencia um aumento contínuo na eficiência operacional do setor aéreo, com crescimento de aproximadamente 54% em 2000 para mais de 80% em 2019.**

- **Esse comportamento indica uma melhoria significativa na utilização da capacidade das aeronaves ao longo do tempo.**

- **Em 2020, observa-se uma queda na taxa de ocupação, refletindo o impacto da pandemia de COVID-19. A recuperação ocorre gradualmente nos anos seguintes, com o setor atingindo níveis recordes de ocupação em 2025.**

- **Pequenas reduções observadas entre 2007 e 2009 podem estar associadas à crise financeira global, que impactou a demanda por transporte aéreo.**

<span style="color:red"><b>Segundo os dados levantados pela ANAC (Agência Nacional de Aviação Civil), o número de passageiros aéreos transportados no Brasil cresceu 11,2 milhões em 2025 ante 2024, para 129,6 milhões, maior demanda já registrada no país. Cerca de 42% dessa alta foi puxada pela Latam. </b></span><br>
**Fonte:** https://www.cnnbrasil.com.br/economia/macroeconomia/anac-latam-responde-por-42-do-crescimento-recorde-da-aviacao-brasileira-em-2025/

In [52]:
# Pandemia e Pós-pandemia
spark.sql("""
SELECT 
    CASE 
        WHEN nr_ano_referencia < 2020 THEN 'pre-pandemia'
        WHEN nr_ano_referencia = 2020 THEN 'pandemia'
        WHEN nr_ano_referencia > 2020 THEN 'pos-pandemia'
    END as periodo,
    AVG(nr_passag_pagos) as media
FROM anac_raw
GROUP BY 
    CASE 
        WHEN nr_ano_referencia < 2020 THEN 'pre-pandemia'
        WHEN nr_ano_referencia = 2020 THEN 'pandemia'
        WHEN nr_ano_referencia > 2020 THEN 'pos-pandemia'
    END
""").show()

[Stage 110:=============================>                          (9 + 8) / 17]

+------------+------------------+
|     periodo|             media|
+------------+------------------+
|pos-pandemia|123.78008960552215|
|pre-pandemia| 95.78658154014978|
|    pandemia| 110.6200781854534|
+------------+------------------+



In [47]:
# Sazonalidade
spark.sql("""
SELECT 
    nr_mes_referencia,
    AVG(nr_passag_pagos) as media
FROM anac_raw
GROUP BY nr_mes_referencia
ORDER BY nr_mes_referencia
""").show()

+-----------------+------------------+
|nr_mes_referencia|             media|
+-----------------+------------------+
|                1|106.06575766706017|
|                2| 98.87642760770005|
|                3| 96.77159973740656|
|                4| 98.74768037184944|
|                5| 96.91028265085635|
|                6| 99.34811360239163|
|                7| 105.5559320075504|
|                8|100.00036314788693|
|                9| 102.2365965245265|
|               10|103.23129359335796|
|               11|103.54606498756387|
|               12|105.70609266735318|
+-----------------+------------------+



**Obs.: Os meses de férias geralmente em Janeiro, Julho, Novembro e Dezembro possuem as médias de passageiros maiores, refletindo exatamente o modo de vida real das pessoas.**

In [61]:
spark.sql("""
SELECT 
    nm_empresa,
    nm_pais,
    ds_tipo_linha,
    AVG(nr_passag_pagos) as media,
    COUNT(*) as qtd
FROM anac_raw
GROUP BY nm_empresa, nm_pais, ds_tipo_linha
HAVING COUNT(*) > 1000
ORDER BY nm_empresa, nm_pais, ds_tipo_linha
""").show(200, truncate=False)

[Stage 131:===================================================>   (16 + 1) / 17]

+-----------------------------------------------------------------------------------+-------------------------+-----------------------+--------------------+-------+
|nm_empresa                                                                         |nm_pais                  |ds_tipo_linha          |media               |qtd    |
+-----------------------------------------------------------------------------------+-------------------------+-----------------------+--------------------+-------+
|ABAETÉ LINHAS AÉREAS S.A.                                                          |BRASIL                   |DOMÉSTICA REGIONAL     |5.169151494331845   |14555  |
|ABSA - AEROLINHAS BRASILEIRAS S.A.                                                 |BRASIL                   |DOMÉSTICA CARGUEIRA    |0.0                 |22991  |
|ABSA - AEROLINHAS BRASILEIRAS S.A.                                                 |BRASIL                   |INTERNACIONAL CARGUEIRA|0.0                 |57203  |
|AEROLINEA

**Obs.: Observa-se a presença de diferentes perfis operacionais no dataset, incluindo companhias de carga, regionais e comerciais. Empresas cargueiras apresentam média de passageiros próxima de zero, enquanto companhias internacionais operam com maior capacidade média, refletindo o uso de aeronaves de maior porte e rotas de longa distância. No Brasil, destaca-se a heterogeneidade do mercado, com coexistência de companhias regionais de baixa capacidade e grandes operadoras nacionais.**

In [67]:
spark.sql("""
SELECT 
    nm_empresa,
    AVG(nr_passag_pagos) as sem_passageiro
FROM anac_raw
WHERE nr_passag_pagos = 0
GROUP BY nm_empresa
ORDER BY nm_empresa
""").show()

+--------------------+--------------+
|          nm_empresa|sem_passageiro|
+--------------------+--------------+
|         21 AIR, LLC|           0.0|
|ABAETÉ LINHAS AÉR...|           0.0|
|ABAETÉ LINHAS AÉR...|           0.0|
|ABSA - AEROLINHAS...|           0.0|
|       ABX AIR, INC.|           0.0|
|ACG AIR CARGO GER...|           0.0|
|        ACT AIRLINES|           0.0|
|AERO V.I.P. LTDA/...|           0.0|
|AEROENLACES NACIO...|           0.0|
|AEROFLOT- AEROLÍN...|           0.0|
|AEROLINEAS ARGENT...|           0.0|
|AEROLÍNEA DEL CAR...|           0.0|
| AERONEXUS CORPORATE|           0.0|
|AEROSERVICES CORP...|           0.0|
|        AEROSUCRE SA|           0.0|
|AEROSUL LINHAS AÉ...|           0.0|
|AEROSUR - CIA BOL...|           0.0|
|  AEROTRANSCARGO SRL|           0.0|
|AEROTRANSPORTE DE...|           0.0|
|AEROTRANSPORTES M...|           0.0|
+--------------------+--------------+
only showing top 20 rows



**Obs.: Essas são algumas das empresas que são cargueiros, por isso não tem passageiros.**

In [71]:
# Removendo linhas que tiverem o nr_passag_pagos = 0, porque são cargueiros e não agrega em nada na previsão
df_limpo = spark.sql("""
SELECT *
FROM anac_raw
WHERE nr_passag_pagos > 0
""")

### Correlação entre a variável dependente(alvo ou Y) e as variáveis independentes (X)

In [72]:
# Selecionando apenas as colunas que possuem valores númericos
colunas_numericas = [
    c for c, t in df.dtypes 
    if t in ('int', 'double', 'float')
]

In [76]:
# Definindo o Target
target = "nr_passag_pagos"
features = [c for c in colunas_numericas if c != target]

In [83]:
resultados = []

for col in features:
    try:
        corr = df.stat.corr(col, target)
        resultados.append((col, corr))
    except:
        pass

# ordenar por importância
resultados_sorted = sorted(resultados, key=lambda x: abs(x[1]), reverse=True)

for col, corr in resultados_sorted:
    print(f"{col}: {corr}")

[Stage 461:====================================================>(149 + 2) / 151]

nr_assentos_ofertados: 0.8206253067910798
nr_rpk: 0.6398808968960765
kg_peso: 0.5938866839627547
nr_ask: 0.5805637950585281
kg_bagagem_livre: 0.5083701585315746
km_distancia: 0.4685559518695994
nr_rtk: 0.42719290168949386
nr_atk: 0.3503388317627833
nr_ano_mes_referencia: 0.3414911501886138
nr_ano_referencia: 0.3413992022473818
id_tipo_linha: -0.3412124668039607
kg_payload: 0.3398555091272217
id_arquivo: 0.32027371057040677
id_basica: 0.29373959676407074
lt_combustivel: 0.2824190704370539
nr_pax_gratis_km: 0.27813363865769974
nr_voo: -0.23566479360397094
nr_passag_gratis: 0.21195423940812697
nr_linha: 0.1740166238213137
nr_etapa: -0.1717096589119938
id_equipamento: -0.1226636556727415
kg_correio: -0.11664693362783089
id_empresa: -0.05604046826694758
nr_bagagem_paga_km: 0.04988069563941085
nr_ano_mes_partida_real: 0.04896003204897099
nr_ano_mes_chegada_real: 0.04895651170258865
nr_ano_partida_real: 0.04894899975887526
nr_ano_chegada_real: 0.04894536723115389
nr_correio_km: 0.037860744354

# **Pré-Processamento**

In [84]:
# Escolhendo as features
features = [
    "nr_assentos_ofertados",
    "km_distancia",
    "kg_payload",
    "lt_combustivel",
    "kg_peso",
    "kg_bagagem_livre",
    "nr_ano_referencia",
    "nr_mes_referencia"
]

**A seleção de variáveis foi realizada com base na correlação de Pearson entre as features numéricas e a variável alvo <mark style="background-color: red; color: white;">nr_passag_pagos</mark>, priorizando aquelas com maior relação linear. No entanto, variáveis derivadas diretamente do target, como <mark style="background-color: #00ff00;">nr_rpk, nr_ask, nr_rtk e nr_atk</mark>, foram removidas para evitar data leakage. Também foram descartados identificadores sem significado analítico, como <mark style="background-color: #00ffff;">id_basica e id_arquivo</mark>. Assim, foram escolhidas variáveis como <mark> nr_assentos_ofertados, kg_peso, kg_bagagem_livre, km_distancia, kg_payload, lt_combustivel, nr_ano_referencia e nr_mes_referencia </mark> por representarem características operacionais relevantes dos voos e apresentarem boa correlação, contribuindo para um modelo mais interpretável e robusto.**

In [86]:
# Criando uma tabela temporária
df_limpo.createOrReplaceTempView("base_limpo")

In [89]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW tabela_features AS
SELECT
    nr_passag_pagos,
    nr_assentos_ofertados,
    km_distancia,
    kg_payload,
    lt_combustivel,
    kg_peso,
    kg_bagagem_livre,
    nr_ano_referencia,
    nr_mes_referencia
FROM base_limpo
""")

DataFrame[]

In [92]:
spark.sql("SELECT * FROM tabela_features").show()

+---------------+---------------------+------------+----------+--------------+-------+----------------+-----------------+-----------------+
|nr_passag_pagos|nr_assentos_ofertados|km_distancia|kg_payload|lt_combustivel|kg_peso|kg_bagagem_livre|nr_ano_referencia|nr_mes_referencia|
+---------------+---------------------+------------+----------+--------------+-------+----------------+-----------------+-----------------+
|          175.0|                216.0|       914.0|   27000.0|        5601.0|14948.0|          1091.0|             2025|               12|
|          210.0|                220.0|       730.0|   27000.0|        4702.0|19248.0|          3317.0|             2025|               12|
|          125.0|                140.0|       928.0|   16531.0|        4526.0|11088.0|           783.0|             2025|               12|
|          115.0|                138.0|       343.0|   16400.0|        2904.0| 9870.0|           945.0|             2025|               12|
|          218.0|   

In [94]:
# Quantidade de dados nulos em cada feature
spark.sql("""
SELECT
    SUM(CASE WHEN nr_passag_pagos IS NULL THEN 1 ELSE 0 END) AS nr_passag_pagos_nulls,
    SUM(CASE WHEN nr_assentos_ofertados IS NULL THEN 1 ELSE 0 END) AS nr_assentos_ofertados_nulls,
    SUM(CASE WHEN km_distancia IS NULL THEN 1 ELSE 0 END) AS km_distancia_nulls,
    SUM(CASE WHEN kg_payload IS NULL THEN 1 ELSE 0 END) AS kg_payload_nulls,
    SUM(CASE WHEN lt_combustivel IS NULL THEN 1 ELSE 0 END) AS lt_combustivel_nulls,
    SUM(CASE WHEN kg_peso IS NULL THEN 1 ELSE 0 END) AS kg_peso_nulls,
    SUM(CASE WHEN kg_bagagem_livre IS NULL THEN 1 ELSE 0 END) AS kg_bagagem_livre_nulls,
    SUM(CASE WHEN nr_ano_referencia IS NULL THEN 1 ELSE 0 END) AS nr_ano_referencia_nulls,
    SUM(CASE WHEN nr_mes_referencia IS NULL THEN 1 ELSE 0 END) AS nr_mes_referencia_nulls
FROM tabela_features
""").show()

[Stage 468:================================>                      (10 + 7) / 17]

+---------------------+---------------------------+------------------+----------------+--------------------+-------------+----------------------+-----------------------+-----------------------+
|nr_passag_pagos_nulls|nr_assentos_ofertados_nulls|km_distancia_nulls|kg_payload_nulls|lt_combustivel_nulls|kg_peso_nulls|kg_bagagem_livre_nulls|nr_ano_referencia_nulls|nr_mes_referencia_nulls|
+---------------------+---------------------------+------------------+----------------+--------------------+-------------+----------------------+-----------------------+-----------------------+
|                    0|                         15|               134|              15|             1235573|            0|               1242381|                      0|                      0|
+---------------------+---------------------------+------------------+----------------+--------------------+-------------+----------------------+-----------------------+-----------------------+



In [101]:
features = [
    "nr_assentos_ofertados",
    "km_distancia",
    "kg_payload",
    "kg_peso",
    "nr_ano_referencia",
    "nr_mes_referencia"
]

**As variáveis `lt_combustivel` e `kg_bagagem_livre` foram removidas do conjunto final de features por apresentarem elevada quantidade de valores nulos, o que poderia reduzir significativamente o volume de dados disponível para modelagem caso fosse aplicada remoção de linhas.**

In [102]:
spark.sql("""
CREATE OR REPLACE TEMP VIEW dataframe_definitivo AS
SELECT
    nr_passag_pagos,
    nr_assentos_ofertados,
    km_distancia,
    kg_payload,
    kg_peso,
    nr_ano_referencia,
    nr_mes_referencia
FROM base_limpo
""")

DataFrame[]

In [103]:
# Excluindo linhas nulos
df_final = spark.table("dataframe_definitivo").dropna()

In [109]:
num_linhas_atualizado = df_final.count()
print(f"Quantidade de linhas no dataset = {num_linhas_atualizado}")

[Stage 477:===>                                                   (1 + 16) / 17]

Quantidade de linhas no dataset = 20955039


**Obs.: Após o tratamento dos dados, temos aproximadamente 21 milhões de linhas.**

In [106]:
# Criando um vetor de features para o modelo
vector_assembler = VectorAssembler(
    inputCols=features,
    outputCol="features"
)

df_vector = vector_assembler.transform(df_final)

df_vector.select("features", target).show(5, truncate=False)

+------------------------------------------+---------------+
|features                                  |nr_passag_pagos|
+------------------------------------------+---------------+
|[216.0,914.0,27000.0,14948.0,2025.0,12.0] |175.0          |
|[220.0,730.0,27000.0,19248.0,2025.0,12.0] |210.0          |
|[140.0,928.0,16531.0,11088.0,2025.0,12.0] |125.0          |
|[138.0,343.0,16400.0,9870.0,2025.0,12.0]  |115.0          |
|[218.0,1920.0,27000.0,18782.0,2025.0,12.0]|218.0          |
+------------------------------------------+---------------+
only showing top 5 rows



In [110]:
# Dividindo em Treino (70%) e Teste (30%)
treino_data, teste_data = df_vector.randomSplit([0.7, 0.3], seed=42)

In [124]:
# Normalizar features númericos, colocando todos na mesma escala para não dar prioridade para o que tiver em uma escala maior
from pyspark.ml.feature import StandardScaler

scaler = StandardScaler(
    inputCol="features",
    outputCol="features_scaled",
    withMean=True,
    withStd=True
)

scaler_model = scaler.fit(treino_data)
treino_scaled = scaler_model.transform(treino_data)
teste_scaled = scaler_model.transform(teste_data)
df_scaled = scaler_model.transform(df_vector)

treino_scaled.select("features_scaled", "nr_passag_pagos").orderBy("nr_passag_pagos", ascending=False).show(10, truncate=False)

[Stage 531:===================================================>   (16 + 1) / 17]

+----------------------------------------------------------------------------------------------------------------------+---------------+
|features_scaled                                                                                                       |nr_passag_pagos|
+----------------------------------------------------------------------------------------------------------------------+---------------+
|[0.4633887182324657,0.6743559008783733,-1.600759493745282,14.853044756271384,-0.1352462600419901,-1.581310341567746]  |1392.0         |
|[0.4633887182324657,0.7659448902159979,-1.600759493745282,14.853044756271384,-0.1352462600419901,-1.581310341567746]  |1392.0         |
|[4.342845544459383,4.34508810607847,4.305499577660142,15.341076804904521,-1.2430677817649705,1.2763298635074471]      |1080.0         |
|[1.797500567975659,2.2126095290307664,20.361901315355784,12.301073362886422,-0.5506793306881077,1.2763298635074471]   |1051.0         |
|[0.4633887182324657,0.1353050419454867,-

# **Modelos de Regressão**

In [148]:
# Criando uma lista vazia para guardar os resultados
resultados = []

## Decision Tree

In [149]:
dt = DecisionTreeRegressor(
    featuresCol="features_scaled",
    labelCol="nr_passag_pagos",
    predictionCol="predicao",
    seed=42 # Controla aleatoriedade, reproduz exatamente o mesmo resultado toda vez que executar
)
# Trieno
modelo_dt = dt.fit(treino_scaled)

In [150]:
# Previsão
pred_dt = modelo_dt.transform(teste_scaled)

pred_dt.select(
    "features_scaled",
    "nr_passag_pagos",
    "predicao"
).show(10, truncate=False)

[Stage 631:>                                                        (0 + 1) / 1]

+-----------------------------------------------------------------------------------------------------------------------+---------------+-----------------+
|features_scaled                                                                                                        |nr_passag_pagos|predicao         |
+-----------------------------------------------------------------------------------------------------------------------+---------------+-----------------+
|[-2.5208088404036246,-0.5218183695069882,-1.4722826410293817,-1.570438406007191,1.664963712757853,1.2763298635074471]  |1.0            |7.488535328334668|
|[-2.50325473711753,-0.3794692174039332,-1.48208571709868,-1.576107703764788,1.664963712757853,0.4190378019848891]      |1.0            |7.488535328334668|
|[-2.50325473711753,-0.3794692174039332,-1.4798234687749958,-1.576107703764788,1.664963712757853,0.9905658429999278]    |1.0            |7.488535328334668|
|[-2.4857006338314354,-0.6338217721307098,-1.4599345355959388,-1

### **Métodos de avaliação para modelos de regressão**

- **RMSE (Root Mean Squared Error)**: Mede o erro médio das previsões, penalizando mais fortemente erros maiores. Quanto menor o valor, melhor o desempenho do modelo.

- **MSE (Mean Squared Error)**: Representa a média dos erros ao quadrado entre os valores previstos e reais. Também penaliza erros maiores e é útil para análise matemática do modelo.

- **MAE (Mean Absolute Error)**: Calcula a média do erro absoluto entre as previsões e os valores reais. É mais interpretável, pois está na mesma unidade da variável alvo.

- **R² (Coeficiente de Determinação)**: Indica o quanto o modelo consegue explicar a variabilidade dos dados. Valores mais próximos de 1 indicam melhor desempenho.

Essas métricas foram utilizadas para comparar os modelos treinados, permitindo identificar qual apresenta melhor capacidade preditiva para o número de passageiros pagos (`nr_passag_pagos`).

In [151]:
# RMSE
evaluator_rmse = RegressionEvaluator(
    labelCol="nr_passag_pagos",
    predictionCol="predicao",
    metricName="rmse"
)
# R²
evaluator_r2 = RegressionEvaluator(
    labelCol="nr_passag_pagos",
    predictionCol="predicao",
    metricName="r2"
)
# MAE
evaluator_mae = RegressionEvaluator(
    labelCol="nr_passag_pagos",
    predictionCol="predicao",
    metricName="mae"
)
# MSE
evaluator_mse = RegressionEvaluator(
    labelCol="nr_passag_pagos",
    predictionCol="predicao",
    metricName="mse"
)

rmse_dt_tree = evaluator_rmse.evaluate(pred_dt)
r2_dt_tree = evaluator_r2.evaluate(pred_dt)
mae_dt_tree = evaluator_mae.evaluate(pred_dt)
mse_dt_tree = evaluator_mse.evaluate(pred_dt)

print(f"RMSE: {rmse_dt_tree}")
print(f"R²: {r2_dt_tree}")
print(f"MAE: {mae_dt_tree}")
print(f"MSE: {mse_dt_tree}")

[Stage 638:===================================================>   (16 + 1) / 17]

RMSE: 15.573178316077016
R²: 0.9255051801564373
MAE: 8.909628038318935
MSE: 242.52388286433134


In [152]:
# Armazenando os resultados em uma lista de dicionario
resultados.append({
    "modelo": "Decision Tree",
    "r2": r2_dt_tree,
    "rmse": rmse_dt_tree,
    "mae": mae_dt_tree,
    "mse": mse_dt_tree
})

## Regressão Linear

In [153]:
lr = LinearRegression(
    featuresCol="features_scaled",
    labelCol="nr_passag_pagos",
    predictionCol="predicao",
    regParam=0.1,
    elasticNetParam=0.0
)

# Treino
modelo_lr = lr.fit(treino_scaled)
# Teste
pred_lr = modelo_lr.transform(teste_scaled)

In [154]:
# RMSE
evaluator_rmse = RegressionEvaluator(
    labelCol="nr_passag_pagos",
    predictionCol="predicao",
    metricName="rmse"
)
# R²
evaluator_r2 = RegressionEvaluator(
    labelCol="nr_passag_pagos",
    predictionCol="predicao",
    metricName="r2"
)
# MAE
evaluator_mae = RegressionEvaluator(
    labelCol="nr_passag_pagos",
    predictionCol="predicao",
    metricName="mae"
)
# MSE
evaluator_mse = RegressionEvaluator(
    labelCol="nr_passag_pagos",
    predictionCol="predicao",
    metricName="mse"
)

rmse_lr_linear = evaluator_rmse.evaluate(pred_lr)
r2_lr_linear = evaluator_r2.evaluate(pred_lr)
mae_lr_linear = evaluator_mae.evaluate(pred_lr)
mse_lr_linear = evaluator_mse.evaluate(pred_lr)

print(f"RMSE: {rmse_lr_linear}")
print(f"R²: {r2_lr_linear}")
print(f"MAE: {mae_lr_linear}")
print(f"MSE: {mse_lr_linear}")

[Stage 650:===================================================>   (16 + 1) / 17]

RMSE: 18.787113428705908
R²: 0.891584425284982
MAE: 12.154683888458317
MSE: 352.9556309830619


In [155]:
# Armazenando os resultados em uma lista de dicionario
resultados.append({
    "modelo": "Linear Regression",
    "r2": r2_lr_linear,
    "rmse": rmse_lr_linear,
    "mae": mae_lr_linear,
    "mse": mse_lr_linear
})

## Redes Neurais

In [157]:
# Exibindo os resultados de cada modelo
df_resultados = df_resultados.select(
    "modelo",
    "mae",
    "mse",
    "rmse",
    "r2"
)

df_resultados.show()

+-----------------+------------------+------------------+------------------+------------------+
|           modelo|               mae|               mse|              rmse|                r2|
+-----------------+------------------+------------------+------------------+------------------+
|    Decision Tree| 8.909628038318935|242.52388286433134|15.573178316077016|0.9255051801564373|
|    Decision Tree| 8.909628038318935|242.52388286433134|15.573178316077016|0.9255051801564373|
|Linear Regression|12.154683888458317| 352.9556309830619|18.787113428705908| 0.891584425284982|
+-----------------+------------------+------------------+------------------+------------------+

